# OverADP historical managed-league simulation

Reproducible analysis of paired ADP, Target Intel, and model-only draft strategies. The primary claim set uses 2023–2024 because those seasons have true Fantasy Football Calculator ADP.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
RESULTS = ROOT / 'results'
raw = pd.read_csv(RESULTS / 'simulation_raw.csv')
summary = pd.read_csv(RESULTS / 'simulation_summary.csv')
paired = pd.read_csv(RESULTS / 'paired_summary.csv')
metadata = json.loads((RESULTS / 'simulation_metadata.json').read_text())
print(raw.shape)
print(summary.to_string(index=False))


## Data and result checks

These checks catch duplicate simulations, incomplete strategy cells, invalid ranks, and missing numeric outputs before interpretation.

In [ ]:
assert raw[['season','episode','strategy']].duplicated().sum() == 0
assert set(raw['strategy']) == {'adp','target_intel','model_only'}
assert raw.groupby(['season','strategy']).size().nunique() == 1
assert raw['regular_rank'].between(1,12).all()
assert np.isfinite(raw[['regular_wins','managed_points','oracle_lineup_points']]).all().all()
assert (raw['oracle_lineup_points'] >= raw['managed_points'] - 1e-9).all()
print('Validation checks passed')


## Primary comparison: true-ADP seasons

Results below pool 2023–2024 only. Pooling gives each paired league equal weight; it does not make the retrospective policy an untouched holdout.

In [ ]:
primary = raw[raw['true_adp']].copy()
primary_summary = (primary.groupby('strategy').agg(
    simulations=('episode','size'),
    avg_rank=('regular_rank','mean'),
    first_place_rate=('regular_rank',lambda s:(s==1).mean()),
    top3_rate=('regular_rank',lambda s:(s<=3).mean()),
    playoff_rate=('made_playoffs','mean'),
    championship_rate=('champion','mean'),
    managed_points=('managed_points','mean'),
).reset_index())
print(primary_summary.to_string(index=False))


In [ ]:
def paired_bootstrap(frame, challenger='target_intel', reps=5000, seed=20260816):
    wide = frame.pivot(index=['season','episode'], columns='strategy', values=['regular_rank','managed_points'])
    rank_delta = (wide['regular_rank']['adp'] - wide['regular_rank'][challenger]).to_numpy()
    points_delta = (wide['managed_points'][challenger] - wide['managed_points']['adp']).to_numpy()
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(rank_delta), size=(reps, len(rank_delta)))
    return {
        'rank_improvement': rank_delta.mean(),
        'rank_improvement_ci_low': np.quantile(rank_delta[idx].mean(axis=1), .025),
        'rank_improvement_ci_high': np.quantile(rank_delta[idx].mean(axis=1), .975),
        'points_delta': points_delta.mean(),
        'points_delta_ci_low': np.quantile(points_delta[idx].mean(axis=1), .025),
        'points_delta_ci_high': np.quantile(points_delta[idx].mean(axis=1), .975),
    }
bootstrap = paired_bootstrap(primary)
print(bootstrap)


## Visual comparison

The chart keeps the 2025 proxy season separate from the two true-ADP seasons.

In [ ]:
plot = summary.copy()
plot['label'] = plot['season'].astype(str) + np.where(plot['true_adp'], ' true ADP', ' proxy')
strategies = ['adp','target_intel','model_only']
labels = plot['label'].drop_duplicates().tolist()
x = np.arange(len(labels)); width = .25
fig, ax = plt.subplots(figsize=(10,5.5))
for offset, strategy in enumerate(strategies):
    values = plot[plot['strategy']==strategy].set_index('label').reindex(labels)['top3_rate']
    ax.bar(x + (offset-1)*width, values, width, label=strategy.replace('_',' ').title())
ax.set_xticks(x, labels); ax.set_ylabel('Top-three regular-season rate'); ax.set_ylim(0, .55)
ax.set_title('Top-three finish rate by strategy and season')
ax.legend(frameon=False); ax.grid(axis='y', alpha=.2)
fig.tight_layout(); fig.savefig(RESULTS / 'top3_rate_by_season.png', dpi=180)
plt.close(fig)
print(RESULTS / 'top3_rate_by_season.png')


## Interpretation boundaries

- Real weekly points naturally include injuries and absences, but the simulator has no injury-designation feed.
- Every team receives the same trailing-information lineup and waiver automation.
- The opponent field is a noisy ADP-and-roster-need approximation rather than observed home-league drafts.
- The strategy was evaluated retrospectively; results support product discovery, not a guaranteed finish or causal advertising claim.
- 2025 is sensitivity evidence only because its market ordering is an ESPN preseason-rank proxy.